# MCXE31 Basic Secure Boot Example

This example demonstrates how to create a signed MBI (Master Boot Image) for MCXE31 devices using Basic Secure Boot mode. The resulting image will be programmed into the internal flash via the `blhost` tool.

## 1. Prerequisites
- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)

- This example uses MCXE31B freedom board
- HSE firmware must be installed on the device

In [1]:
import os

from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)
MBI_CONFIG = (
    INPUTS + "mcxe31_mbi_config.yaml"
)  # MBI configuration file. Used to generate signed MBI image

SIGNED_MBI = WORKSPACE + "mcxe31_mbi.bin"  # Final signed MBI image
ADKP_KEY = INPUTS + "adkp.bin"  # ADKP key for provisioning

HSE_FW = INPUTS + "mcxe3xb_hse_fw_0.5.0_2.55.0_pb250324.bin.pink"  # HSE firmware

# Flash addresses
APP_ADDRESS = "0x400000"
VERIFY_ADDRESS = "0x00400fC0"

# Communication parameters
BLHOST_CONNECT = "-p COM21"  # COM port - change this to your actual COM port

### 1.1 Compile an Application
Blinky LED XiP application is used in this example. 

To use a custom application follow the steps:

- Download the NXP MCUXpresso SDK from NXP web site (https://mcuxpresso.nxp.com/)
- Open the example in your favorite IDE. 
- Compile your application using the corresponding target. The resulting binary must be copied to input folder.

## 2. Basic Secure Boot Overview

The MCXE31 Basic Secure Boot (also known as IVT-based secure boot) mode is implemented based on the AppBL header and ADKP, and the HSE firmware enables only one application core at a time. The length of AppBL header is 64 bytes and the application
code address starts from “AppBL header start address + AppBL header length”.

Key features of Basic Secure Boot:

- **Single Core Operation**: Only one application core is enabled, as specified in the Image Vector Table (IVT)
- **GMAC Authentication**: Uses GMAC (Galois Message Authentication Code) based authentication with a key derived from the Application Debug Key Provisioning (ADKP)
- **Mutual Exclusivity**: Can only be used when SMR/CR (Secure Memory Region/Code Region) based Secure Boot is not in use
- **IVT Structure**: The boot process relies on the Image Vector Table to define the application entry point and authentication parameters

This approach provides a balance between security and simplicity, making it suitable for applications that require authenticated boot without the complexity of advanced secure boot mechanisms.

## 3. Initialize Flashloader

The MCX-E Flashloader is a configurable flash programming utility that operates over a serial connection on MCX-E device. It enables quick and easy programming of MCUs during final product manufacturing.

Before we can communicate with the device, we need to initialize the flashloader using J-Link. In this example, we'll use the J-Link debugger to establish communication.


| Interface | Pin function    |    Pin   |
| --------- | --------------- | -------- |
| LPUART    | LPUART0_TX      | PTA3     |
|           | LPUART0_RX      | PTA2     |
| LPSPI     | LPSPI0_MOSI     | PTB1     |
|           | LPSPI0_SCK      | PTC8     |
|           | LPSPI0_MISO     | PTC9     |
|           | LPSPI0_PCS0     | PTB0     |



In [3]:
# Initialize flashloader using J-Link
# Make sure J-Link is connected to the board and the board is powered on
%! "JLink.exe" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/init_flashloader_cmd.jlink"

"JLink.exe" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/init_flashloader_cmd.jlink" 
SEGGER J-Link Commander V8.44 (Compiled Jun 18 2025 12:17:39)
DLL version V8.44, compiled Jun 18 2025 12:16:46

J-Link Commander will now exit on Error

J-Link Command File read successfully.
Processing script file...
J-Link>device S32K314
J-Link connection not established yet but required for command.
Connecting to J-Link via USB...O.K.
Firmware: J-Link V12 compiled Sep 17 2025 12:01:50
Hardware version: V12.00
J-Link uptime (since boot): 0d 00h 00m 16s
S/N: 602010037
License(s): RDI, FlashBP, FlashDL, JFlash, GDB
USB speed mode: High speed (480 MBit/s)
VTref=3.303V
J-Link>si SWD
Selecting SWD as current target interface.
J-Link>speed 4000
Selecting 4000 kHz as target interface speed
J-Link>connect
Device "S32K314" selected.


Connecting to target via SWD
ConfigTargetSettings() start
ConfigTargetSettings() end - Took 19us
InitTarget() start
SDA_AP detected
Unlocking device if necessary...
  Device is

## 4. Install HSE Firmware (Optional)

All MCXE31 devices are delivered from factory without security firmware, as a measure to ensure a chain
of trust and guarantee that only authorized parties get the security firmware. Therefore, the first step is to
install HSE Firmware to the device.

The HSE FW installation is a one-time process, and the HSE FW can only be updated and can’t be uninstalled after installation.

If HSE firmware is not already installed, uncomment and run the following commands:

In [4]:
# Uncomment the following lines if HSE firmware needs to be installed
# assert os.path.exists(HSE_FW)
# %! blhost $BLHOST_CONNECT flash-erase-region 0x420000 0x40000
# %! blhost $BLHOST_CONNECT --timeout 120 write-memory 0x420000S $HSE_FW
# %! blhost $BLHOST_CONNECT --timeout 120 write-memory 0x1b000000 "{{ AA BB CC DD DD CC BB AA }}"

## 5. Check HSE Firmware version

Let's verify that we can communicate with the device and check the HSE firmware version. If the previous step was successful, the device will respond with valid version number. In our case it is H2.55.0.

In [5]:
# Check device properties
%! blhost $BLHOST_CONNECT get-property 24 1

blhost -p COM21 get-property 24 1 
Response status = 0 (0x0) Success.
Response word 1 = 1208104704 (0x48023700)
Target Version = H2.55.0


## 6. Provision ADKP Key

The Application Debug Key/Password (ADKP) is an HSE OTP (one-time program) attribute, not like a normal NVM key which can be erased. 
It is used to calculate the GMAC of application of 16 bytes through AES-GMAC algorithm in RAM, based on a 256-bit AES key generated from SHA256 operation over the user-defined ADKP

In order to provision Application Debug Key/Password (ADKP) key, run the code bellow.

In [ ]:
assert os.path.exists(ADKP_KEY)

# Enroll key provisioning
%! blhost $BLHOST_CONNECT key-provisioning enroll

# Set user key
%! blhost $BLHOST_CONNECT key-provisioning set_user_key 0 $ADKP_KEY

blhost -p COM21 key-provisioning enroll
Response status = 0 (0x0) Success.
blhost -p COM21 key-provisioning set_user_key 0 inputs/adkp.bin
Response status = 0 (0x0) Success.


## 7. Create Signed MBI Container

Now we'll create a signed MBI container for our application. First, let's generate the MBI configuration template.

In [7]:
# Show configuration differences
YamlDiffWidget("inputs/mcxe31_mbi_config.diffc").html

nxpimage mbi get-templates -f mcxe31b -o workspace --force 
Creating workspace/mcxe31b_xip_plain.yaml template file.
Creating workspace/mcxe31b_xip_signed.yaml template file.


### 7.1 Create signed masterboot image

In [ ]:
assert os.path.exists(MBI_CONFIG)

# Export signed MBI image using nxpimage
%! nxpimage mbi export -c $MBI_CONFIG

assert os.path.exists(SIGNED_MBI)

nxpimage mbi export -c inputs/mcxe31_mbi_config.yaml 
Success. (Master Boot Image: workspace/mcxe31_mbi.bin created.)


## 8. Board Setup

For this example we will be using MCXE31B Freedom board. For getting familiar with the board, visit this [link](https://www.nxp.com/document/guide/getting-started-with-the-frdm-mcxe31b-board:GS-FRDM-MCXE31B).

Ensure your MCXE31B Freedom board is configured as follows:

- **J-Link Connection**: Connect J-Link debugger to the board
- **UART Connection**: Connect UART cable to the appropriate COM port. In our case, we connected USB to UART converter to pins PTA2 and PTA3
- **Power**: Ensure the board is properly powered via USB


<img src="../../_data/img/boards/frdm-mcxe31B.png" alt="frdm-mcxe31B" height="300">

## 9. Program Signed MBI Image

Now let's program the signed MBI image to the device:

In [9]:
# Erase flash region
%! blhost $BLHOST_CONNECT flash-erase-region $APP_ADDRESS 0x40000

# Program signed MBI image
%! blhost $BLHOST_CONNECT write-memory $APP_ADDRESS $SIGNED_MBI

blhost -p COM21 flash-erase-region 0x400000 0x40000 
Response status = 0 (0x0) Success.
blhost -p COM21 write-memory 0x400000 workspace/mcxe31_mbi.bin 
Writing memory
Response status = 0 (0x0) Success.
Response word 1 = 19596 (0x4c8c)


## 10. Verify Image Signature

Finally, let's verify that the image signature is valid using the HSE firmware:

In [10]:
# Verify image signature using HSE
%! nxpele -f mcxe31b $BLHOST_CONNECT hse img-verify -a $VERIFY_ADDRESS

nxpele -f mcxe31b -p COM21 hse img-verify -a 0x00400fC0 
Boot Data Image verification successful


## 11. Test the Application

After programming the signed image, you can:

1. **Reset the board** to start the application
2. **Check the LED behavior** - the red LED should blink according to the application logic

The signed MBI container ensures that:
- Only authenticated firmware can run on the device
- The image integrity is verified during boot
- The HSE firmware validates the signature before execution

## Summary

This example demonstrated the complete flow for MCXE31 basic secure boot:

1. **Device Preparation**: Initialized flashloader and verified communication
2. **Key Provisioning**: Enrolled and set the ADKP key for secure operations
3. **MBI Creation**: Generated a signed Master Boot Image
4. **Image Programming**: Programmed signed images to internal flash
5. **Signature Verification**: Used HSE firmware to verify the image signature

The secure boot process ensures that only properly signed firmware can execute on the MCXE31 device, providing a foundation for a secure system.